In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns # figs & visualisiton
import kagglehub
import os
from tqdm import tqdm # the loop bar

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)

df = pd.read_csv(delivery_path)

print(f"Dataset shape: {df_delivery.shape}")

In [ ]:
# Task 2: Write your code here:
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
#delivery_time distribution
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='lightBlue')
plt.title('Delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df_delivery.drop(columns=['Order_ID']).copy()
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_delivery.isnull().sum())

In [ ]:
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs' ]

df_clean = df_delivery.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")

#df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna('None')
df_mean = df.copy()
df_mean['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
feature_cols = ['Courier_Experience_yrs',	'Delivery_Time', 'Traffic_Level',	'Time_of_Day', 'Distance_km' ]
X = df_clean[feature_cols]
y = df_clean[df_mean.copy]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Delivery time in train in train: {y_train.sum()}, in test: {y_test.sum()}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()

In [ ]:
# Task 2,3,4,5: Write your code here:
#StratifiedKFold
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(model, X, y, cv=skfold, scoring='accuracy')
print("\nStratifiedKFold scores:", scores_strat)
print(f"Mean accuracy: {scores_strat.mean():.3f}")

In [ ]:
# Task 1: Write your code here:
importances = {}

importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
cm = confusion_matrix(y_test,y_pred)

plt.figure(figsize=(6, 4))
plt.imshow(cm, cmap='Pastel1')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks([0, 1], ['Delivery time', 'distance'])
plt.yticks([0, 1], ['Delivery time', 'distance'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='red')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Task Bonus: Write your code here: